In [1]:
# Import library
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier

print("pandas:", pd.__version__)
import sklearn
print("scikit-learn:", sklearn.__version__)


pandas: 3.0.6
scikit-learn: 1.9.1


In [2]:
# Gunakan dataset hasil clustering yang memiliki fitur Target
df = pd.read_csv("data_clustering.csv")
print("Shape:", df.shape)
print("Kolom:", df.columns.tolist())
assert "Target" in df.columns
print("Distribusi Target:")
print(df["Target"].value_counts().sort_index())


Shape: (2358, 13)
Kolom: ['TransactionAmount', 'AccountAgeDays', 'NumTransactionsLast7D', 'AvgTransactionAmount7D', 'TransactionHour', 'BalanceBefore', 'BalanceAfter', 'TransactionType', 'MerchantCategory', 'DeviceType', 'Location', 'AmountBin', 'Target']
Distribusi Target:
Target
0    860
1    312
2    635
3    551
Name: count, dtype: int64


In [3]:
# Tampilkan 5 baris pertama dengan function head
df.head()


,TransactionAmount,AccountAgeDays,NumTransactionsLast7D,AvgTransactionAmount7D,TransactionHour,BalanceBefore,BalanceAfter,TransactionType,MerchantCategory,DeviceType,Location,AmountBin,Target
0,-0.712759,-0.331391,1.529329,-0.740189,1.383061,1.153266,1.187845,1,2,1,3,0,2
1,-0.785220,-0.538882,1.327834,-0.802968,0.129454,-0.636996,-0.596553,0,3,1,2,0,0
2,0.377638,-0.987014,-0.687117,0.209539,-1.594255,-1.059461,-1.077230,2,1,1,0,2,3
3,1.343579,-1.312369,-1.090107,1.487604,-1.437554,0.137492,0.069463,2,4,1,3,2,1
4,-0.765086,-0.053918,-0.082632,-0.759650,-0.654050,0.383160,0.421330,0,3,0,0,0,0


### MULAI CODE OPSIONAL ###

In [4]:
# Menggunakan train_test_split() untuk melakukan pembagian dataset.
X = df.drop(columns=["Target"])
y = df["Target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Distribusi label train:")
print(y_train.value_counts(normalize=True).round(3))
print("Distribusi label test:")
print(y_test.value_counts(normalize=True).round(3))


X_train: (1886, 12) | X_test: (472, 12)
Distribusi label train:
Target
0    0.365
2    0.269
3    0.234
1    0.132
Name: proportion, dtype: float64
Distribusi label test:
Target
0    0.364
2    0.269
3    0.233
1    0.133
Name: proportion, dtype: float64


In [5]:
# Buatlah model klasifikasi menggunakan Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print("Decision Tree - Akurasi test:", round(accuracy_score(y_test, y_pred_dt), 4))
print("Decision Tree - F1-Score (weighted) test:", round(f1_score(y_test, y_pred_dt, average="weighted"), 4))
print(classification_report(y_test, y_pred_dt))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_dt))


Decision Tree - Akurasi test: 0.9767
Decision Tree - F1-Score (weighted) test: 0.9767
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       172
           1       0.95      0.95      0.95        63
           2       0.98      0.96      0.97       127
           3       0.96      0.99      0.98       110

    accuracy                           0.98       472
   macro avg       0.97      0.97      0.97       472
weighted avg       0.98      0.98      0.98       472

Confusion matrix:
[[170   1   0   1]
 [  1  60   2   0]
 [  0   2 122   3]
 [  0   0   1 109]]


In [6]:
# Menyimpan Model
joblib.dump(dt, "decision_tree_model.h5")
print("Model tersimpan: decision_tree_model.h5")
print("Tipe objek:", type(joblib.load("decision_tree_model.h5")))


Model tersimpan: decision_tree_model.h5
Tipe objek: <class 'sklearn.tree._classes.DecisionTreeClassifier'>


In [7]:
# Melatih model menggunakan algoritma klasifikasi scikit-learn selain Decision Tree. (Contoh: RandomForestClassifier)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest - Akurasi test:", round(accuracy_score(y_test, y_pred_rf), 4))
print("Random Forest - F1-Score (weighted) test:", round(f1_score(y_test, y_pred_rf, average="weighted"), 4))


Random Forest - Akurasi test: 0.9767
Random Forest - F1-Score (weighted) test: 0.9767


In [8]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada seluruh algoritma yang sudah dibuat.
results = pd.DataFrame([
    {"model": "Decision Tree",
     "accuracy": accuracy_score(y_test, y_pred_dt),
     "precision": precision_score(y_test, y_pred_dt, average="weighted", zero_division=0),
     "recall": recall_score(y_test, y_pred_dt, average="weighted", zero_division=0),
     "f1": f1_score(y_test, y_pred_dt, average="weighted")},
    {"model": "Random Forest",
     "accuracy": accuracy_score(y_test, y_pred_rf),
     "precision": precision_score(y_test, y_pred_rf, average="weighted", zero_division=0),
     "recall": recall_score(y_test, y_pred_rf, average="weighted", zero_division=0),
     "f1": f1_score(y_test, y_pred_rf, average="weighted")},
]).round(4)
display(results)
print("Classification report Decision Tree:")
print(classification_report(y_test, y_pred_dt))
print("Classification report Random Forest:")
print(classification_report(y_test, y_pred_rf))


,model,accuracy,precision,recall,f1
0,Decision Tree,0.9767,0.9768,0.9767,0.9767
1,Random Forest,0.9767,0.9767,0.9767,0.9767


Classification report Decision Tree:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       172
           1       0.95      0.95      0.95        63
           2       0.98      0.96      0.97       127
           3       0.96      0.99      0.98       110

    accuracy                           0.98       472
   macro avg       0.97      0.97      0.97       472
weighted avg       0.98      0.98      0.98       472

Classification report Random Forest:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       172
           1       0.95      0.95      0.95        63
           2       0.97      0.97      0.97       127
           3       0.97      0.98      0.98       110

    accuracy                           0.98       472
   macro avg       0.97      0.97      0.97       472
weighted avg       0.98      0.98      0.98       472



In [9]:
# Menyimpan Model Selain Decision Tree
joblib.dump(rf, "explore_RandomForest_classification.h5")
print("Model tersimpan: explore_RandomForest_classification.h5")
print("Tipe objek:", type(joblib.load("explore_RandomForest_classification.h5")))


Model tersimpan: explore_RandomForest_classification.h5
Tipe objek: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [10]:
# Lakukan Hyperparameter Tuning dan Latih ulang.
param_grid = {
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"],
}
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV F1 (weighted):", round(float(grid.best_score_), 4))
best_dt = grid.best_estimator_
y_pred_tuned = best_dt.predict(X_test)
print("Tuned DT - Akurasi test:", round(accuracy_score(y_test, y_pred_tuned), 4))
print("Tuned DT - F1-Score (weighted) test:", round(f1_score(y_test, y_pred_tuned, average="weighted"), 4))


Best params: {'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 10}
Best CV F1 (weighted): 0.9702
Tuned DT - Akurasi test: 0.9746
Tuned DT - F1-Score (weighted) test: 0.9745


In [11]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.
print("Classification report Tuned Decision Tree (test):")
print(classification_report(y_test, y_pred_tuned))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_tuned))
final_results = pd.DataFrame([
    {"model": "Decision Tree",
     "accuracy": accuracy_score(y_test, y_pred_dt),
     "precision": precision_score(y_test, y_pred_dt, average="weighted", zero_division=0),
     "recall": recall_score(y_test, y_pred_dt, average="weighted", zero_division=0),
     "f1": f1_score(y_test, y_pred_dt, average="weighted")},
    {"model": "Random Forest",
     "accuracy": accuracy_score(y_test, y_pred_rf),
     "precision": precision_score(y_test, y_pred_rf, average="weighted", zero_division=0),
     "recall": recall_score(y_test, y_pred_rf, average="weighted", zero_division=0),
     "f1": f1_score(y_test, y_pred_rf, average="weighted")},
    {"model": "Tuned Decision Tree",
     "accuracy": accuracy_score(y_test, y_pred_tuned),
     "precision": precision_score(y_test, y_pred_tuned, average="weighted", zero_division=0),
     "recall": recall_score(y_test, y_pred_tuned, average="weighted", zero_division=0),
     "f1": f1_score(y_test, y_pred_tuned, average="weighted")},
]).round(4)
display(final_results)


Classification report Tuned Decision Tree (test):
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       172
           1       0.95      0.92      0.94        63
           2       0.97      0.98      0.97       127
           3       0.98      0.98      0.98       110

    accuracy                           0.97       472
   macro avg       0.97      0.97      0.97       472
weighted avg       0.97      0.97      0.97       472

Confusion matrix:
[[170   1   0   1]
 [  3  58   2   0]
 [  0   2 124   1]
 [  0   0   2 108]]


,model,accuracy,precision,recall,f1
0,Decision Tree,0.9767,0.9768,0.9767,0.9767
1,Random Forest,0.9767,0.9767,0.9767,0.9767
2,Tuned Decision Tree,0.9746,0.9745,0.9746,0.9745


In [12]:
# Menyimpan Model hasil tuning
joblib.dump(best_dt, "tuning_classification.h5")
print("Model tersimpan: tuning_classification.h5")
print("Tipe objek:", type(joblib.load("tuning_classification.h5")))


Model tersimpan: tuning_classification.h5
Tipe objek: <class 'sklearn.tree._classes.DecisionTreeClassifier'>
